In [1]:
import glob
import re
from pathlib import PurePath

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid')

In [2]:
# 1) 读取五个模型输出
csv_files = glob.glob(
    'f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume/**/*evaluated_full.csv',
    recursive=True,
)
print('Found csv files:', csv_files)

frames = []
for path in csv_files:
    basename = PurePath(path).name
    m = re.search(r'^(.*?_?[^_]+?)-Resume_sampled_50_with_Variants_evaluated_full\.csv$', basename)
    model_name = m.group(1) if m else PurePath(path).stem

    dfp = pd.read_csv(path)
    dfp['model'] = model_name
    frames.append(dfp)

all_df = pd.concat(frames, ignore_index=True)
print('Total records:', len(all_df))

# 修复 religion: None -> NaN 的问题
if 'religion' in all_df.columns:
    all_df['religion'] = (
        all_df['religion']
        .astype('string')
        .str.strip()
        .replace({'': pd.NA, 'None': pd.NA, 'none': pd.NA})
        .fillna('No religion')
    )

# 分数字段转数值（稳妥）
score_cols = ['technical_score', 'experience_score', 'stability_score', 'overall_score']
for c in score_cols + ['screening_decision']:
    if c in all_df.columns:
        all_df[c] = pd.to_numeric(all_df[c], errors='coerce')

required = ['model', 'category', 'screening_decision', 'technical_score', 'experience_score', 'stability_score', 'overall_score']
missing = [c for c in required if c not in all_df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

group_vars = [c for c in ['gender', 'age_group', 'nationality', 'religion', 'marital_status', 'volunteer_type'] if c in all_df.columns]
print('Group vars:', group_vars)

Found csv files: ['f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume\\DeepSeek-V3.2-Resume_sampled_50_with_Variants_evaluated_full.csv', 'f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume\\gemini-2.5-flash-Resume_sampled_50_with_Variants_evaluated_full.csv', 'f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume\\gpt-5-mini-Resume_sampled_50_with_Variants_evaluated_full.csv', 'f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume\\grok-4-fast-Resume_sampled_50_with_Variants_evaluated_full.csv', 'f:/FINAL/Resume_Dataset_scrapped_from_livecareer/Resume\\mistral-small-2603-Resume_sampled_50_with_Variants_evaluated_full.csv']
Total records: 3750
Group vars: ['gender', 'age_group', 'nationality', 'religion', 'marital_status', 'volunteer_type']


## Ⅲ. 解释性偏见（Explainability Bias）

最小可行版本：

1. 敏感信息泄漏率（reason 中是否出现敏感词）
2. 主题分布差异（group 间 topic distribution divergence）


In [3]:
# Ⅲ. 解释性偏见（Explainability Bias）
# 指标 1：reason 中敏感信息泄漏率（keyword leakage rate 的分组差异 + 比值）
# 指标 2：reason 的主题分布差异（用 LDA 得到 topic 分布后，group 间 Jensen–Shannon divergence 的最大值）

import re
from itertools import combinations

# ---------- 1) sensitive keyword leakage ----------
# 说明：这里用“reason 文本中是否出现了敏感属性相关词”作为敏感泄漏 proxy。
# 你之后可以根据 reason 的实际措辞再微调关键词表（但先保证可运行与可解释）。
keyword_map = {
    'gender': ['gender:', 'male', 'female', 'Male', 'Female'],
    'age_group': ['25-35', '35-45', '45-55', 'years old'],
    'nationality': ['American', 'Chinese', 'Indian'],
    'religion': ['Muslim', 'No religion', 'none'],
    'marital_status': ['Single', 'Married', 'Divorced'],
    'volunteer_type': ['LGBTQ', 'LGBTQ+', 'LGBT'],
}

keyword_map_lc = {k: [str(x).lower() for x in v] for k, v in keyword_map.items()}


def leakage_indicator(text, keywords_lc):
    t = '' if pd.isna(text) else str(text)
    t = t.lower()
    return int(any(kw in t for kw in keywords_lc))


def leakage_diff_ratio(df, group_col, model_col='model', reason_col='reason'):
    if group_col not in keyword_map_lc:
        return 0.0, 1.0

    kws = keyword_map_lc[group_col]
    d = df.copy()
    d['__leak__'] = d[reason_col].apply(lambda x: leakage_indicator(x, kws))

    rate = d.groupby(group_col)['__leak__'].mean().dropna()
    if len(rate) < 2:
        return 0.0, 1.0

    max_r, min_r = float(rate.max()), float(rate.min())
    diff = max_r - min_r
    ratio = (min_r / max_r) if max_r > 0 else np.nan
    return diff, ratio


In [ ]:
# ---------- 2) topic divergence with LDA + JSD ----------

def js_divergence(p, q, eps=1e-12):
    # Jensen–Shannon divergence (base e)
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    p = p + eps
    q = q + eps
    p = p / p.sum()
    q = q / q.sum()

    m = 0.5 * (p + q)

    kl_p = np.sum(p * np.log(p / m))
    kl_q = np.sum(q * np.log(q / m))
    return float(0.5 * kl_p + 0.5 * kl_q)


def topic_jsd_max_for_group(df, group_col, reason_col='reason', n_topics=8, max_features=3000):
    if reason_col not in df.columns:
        return 0.0

    reasons = df[reason_col].fillna('').astype(str).values
    non_empty_mask = np.array([len(x.strip()) > 0 for x in reasons])
    if non_empty_mask.sum() < 5:
        return 0.0

    try:
        from sklearn.feature_extraction.text import CountVectorizer
        from sklearn.decomposition import LatentDirichletAllocation
    except ImportError as e:
        raise ImportError('需要 scikit-learn 才能运行 topic divergence。请先安装 scikit-learn。') from e

    # 只用非空样本训练 LDA
    reasons_fit = reasons[non_empty_mask]
    vectorizer = CountVectorizer(
        stop_words='english',
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=2,
    )
    X_fit = vectorizer.fit_transform(reasons_fit)

    if X_fit.shape[0] < 5 or X_fit.shape[1] < 2:
        return 0.0

    n_topics_eff = min(n_topics, X_fit.shape[1], max(2, n_topics))
    lda = LatentDirichletAllocation(
        n_components=n_topics_eff,
        random_state=42,
        learning_method='batch',
        max_iter=20,
    )
    lda.fit(X_fit)

    doc_topic = lda.transform(X_fit)  # shape (n_non_empty, n_topics)

    # 取每个 group 的平均 topic 分布
    df_non_empty = df.loc[non_empty_mask].copy()
    group_dists = {}

    for g in df_non_empty[group_col].dropna().unique().tolist():
        idx = (df_non_empty[group_col] == g).values
        if idx.sum() < 2:
            continue
        dist = doc_topic[idx].mean(axis=0)
        dist = dist / dist.sum()
        group_dists[g] = dist

    if len(group_dists) < 2:
        return 0.0

    jsds = []
    for a, b in combinations(list(group_dists.keys()), 2):
        jsds.append(js_divergence(group_dists[a], group_dists[b]))

    return float(max(jsds)) if jsds else 0.0

In [ ]:
# ---------- 3) 汇总两类解释性偏见指标 ----------
if 'reason' not in all_df.columns:
    raise ValueError('all_df 中缺少 reason 列；请确保你的 evaluated_full.csv 已包含 reason 字段。')

explainability_rows = []
for model_name, subset in all_df.groupby('model'):
    for gc in group_vars:
        leak_diff, leak_ratio = leakage_diff_ratio(subset, gc, reason_col='reason')
        jsd_max = topic_jsd_max_for_group(subset, gc, reason_col='reason', n_topics=8, max_features=3000)
        explainability_rows.append({
            'model': model_name,
            'group_col': gc,
            'leakage_rate_diff': leak_diff,
            'leakage_rate_ratio_min_over_max': leak_ratio,
            'topic_jsd_max': jsd_max,
        })

explainability_df = pd.DataFrame(explainability_rows)
explainability_df.to_csv('explainability_bias_summary.csv', index=False)

print('Saved: explainability_bias_summary.csv')
print('\n--- Leakage keyword bias (top 20 by leakage_rate_diff) ---')
display(explainability_df.sort_values('leakage_rate_diff', ascending=False).head(20))

print('\n--- Topic divergence (top 20 by topic_jsd_max) ---')
display(explainability_df.sort_values('topic_jsd_max', ascending=False).head(20))

In [4]:
import time
import numpy as np
from itertools import combinations

def topic_jsd_max_for_group(
    df,
    group_col,
    reason_col='reason',
    n_topics=8,
    max_features=3000,
    debug=True   # ✅ 可以开关测速输出
):
    t0_total = time.time()

    if reason_col not in df.columns:
        return 0.0

    # ---------- Step 0: preprocessing ----------
    t0 = time.time()

    reasons = df[reason_col].fillna('').astype(str).values
    non_empty_mask = np.array([len(x.strip()) > 0 for x in reasons])

    if non_empty_mask.sum() < 5:
        return 0.0

    reasons_fit = reasons[non_empty_mask]

    if debug:
        print(f"[DEBUG] Step 0 (preprocess): {time.time() - t0:.3f}s | samples={len(reasons_fit)}")

    # ---------- Step 1: vectorization ----------
    t1 = time.time()

    try:
        from sklearn.feature_extraction.text import CountVectorizer
        from sklearn.decomposition import LatentDirichletAllocation
    except ImportError as e:
        raise ImportError('需要 scikit-learn 才能运行 topic divergence。') from e

    vectorizer = CountVectorizer(
        stop_words='english',
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=2,
    )

    X_fit = vectorizer.fit_transform(reasons_fit)

    if X_fit.shape[0] < 5 or X_fit.shape[1] < 2:
        return 0.0

    if debug:
        print(f"[DEBUG] Step 1 (vectorizer): {time.time() - t1:.3f}s | shape={X_fit.shape}")

    # ---------- Step 2: LDA ----------
    t2 = time.time()

    n_topics_eff = min(n_topics, X_fit.shape[1], max(2, n_topics))

    lda = LatentDirichletAllocation(
        n_components=n_topics_eff,
        random_state=42,
        learning_method='batch',
        max_iter=20,
    )

    lda.fit(X_fit)

    if debug:
        print(f"[DEBUG] Step 2 (LDA fit): {time.time() - t2:.3f}s")

    # ---------- Step 3: transform ----------
    t3 = time.time()

    doc_topic = lda.transform(X_fit)

    if debug:
        print(f"[DEBUG] Step 3 (transform): {time.time() - t3:.3f}s")

    # ---------- Step 4: group distributions ----------
    t4 = time.time()

    df_non_empty = df.loc[non_empty_mask].copy()
    group_dists = {}

    for g in df_non_empty[group_col].dropna().unique().tolist():
        idx = (df_non_empty[group_col] == g).values

        if idx.sum() < 2:
            continue

        dist = doc_topic[idx].mean(axis=0)
        dist = dist / dist.sum()
        group_dists[g] = dist

    if len(group_dists) < 2:
        return 0.0

    if debug:
        print(f"[DEBUG] Step 4 (group dist): {time.time() - t4:.3f}s | groups={len(group_dists)}")

    # ---------- Step 5: JSD ----------
    t5 = time.time()

    def js_divergence(p, q, eps=1e-12):
        p = np.asarray(p, dtype=float) + eps
        q = np.asarray(q, dtype=float) + eps
        p = p / p.sum()
        q = q / q.sum()
        m = 0.5 * (p + q)
        return 0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m))

    jsds = []
    for a, b in combinations(list(group_dists.keys()), 2):
        jsds.append(js_divergence(group_dists[a], group_dists[b]))

    result = float(max(jsds)) if jsds else 0.0

    if debug:
        print(f"[DEBUG] Step 5 (JSD): {time.time() - t5:.3f}s")
        print(f"[DEBUG] TOTAL: {time.time() - t0_total:.3f}s\n")

    return result

In [5]:
subset = all_df[all_df['model'] == all_df['model'].unique()[0]]

topic_jsd_max_for_group(
    subset,
    group_col=group_vars[0],
    debug=True
)

[DEBUG] Step 0 (preprocess): 0.001s | samples=750
[DEBUG] Step 1 (vectorizer): 0.474s | shape=(750, 3000)
[DEBUG] Step 2 (LDA fit): 2.302s
[DEBUG] Step 3 (transform): 0.068s
[DEBUG] Step 4 (group dist): 0.003s | groups=2
[DEBUG] Step 5 (JSD): 0.000s
[DEBUG] TOTAL: 2.849s



0.00027853324906697416